# XGBoost — Terrenos (Optuna)

Modelo XGBoost optimizado con Optuna sobre `data/gold/final_land_scraping.csv`, como referencia de rendimiento frente a los modelos lineales de `51_linear_regression_def.ipynb`.

**Contexto:** con solo 7 features y una feature principal prácticamente sin señal (`superficie_m2`, r=0.07), la ventaja de XGBoost sobre Ridge es menos clara que en los datasets de vivienda. El notebook confirma empíricamente si las interacciones no lineales (`municipio × tipo_suelo`) añaden valor sobre el modelo lineal.

**Target:** `log_precio`. Métricas finales también en €.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
from scipy import stats
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Carga y preparación

In [ ]:
df = pd.read_csv('../../data/gold/final_land_scraping.csv')

# XGBoost no tolera espacios ni paréntesis en nombres de columnas
df.columns = (
    df.columns
    .str.replace(' ', '_', regex=False)
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)

TARGET   = 'log_precio'
EXCLUDE  = ['precio_eur', TARGET]
FEATURES = [c for c in df.columns if c not in EXCLUDE]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Shape dataset: {df.shape}')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features ({len(FEATURES)}): {FEATURES}')

## 2. Función de métricas

In [ ]:
def get_metrics(model, X_tr, X_te, y_tr, y_te, name=''):
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    m = {
        'RMSE_train': np.sqrt(mean_squared_error(y_tr, y_pred_tr)),
        'RMSE_test':  np.sqrt(mean_squared_error(y_te, y_pred_te)),
        'R2_train':   r2_score(y_tr, y_pred_tr),
        'R2_test':    r2_score(y_te, y_pred_te),
        'MAE_test':   mean_absolute_error(y_te, y_pred_te),
    }
    if name:
        print(f'--- {name} ---')
        for k, v in m.items():
            print(f'  {k}: {v:.4f}')
    return m

## 3. XGBoost baseline

Modelo con hiperparámetros por defecto para establecer el punto de partida antes de optimizar.

In [ ]:
xgb_base = XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
xgb_base.fit(X_train, y_train)

metrics_base = get_metrics(xgb_base, X_train, X_test, y_train, y_test, name='XGBoost baseline')

# Overfitting check
gap = metrics_base['RMSE_train'] - metrics_base['RMSE_test']
print(f'\nGap RMSE (train - test): {gap:.4f}')
if abs(gap) > 0.05:
    print('  -> Overfitting detectado — la regularización de Optuna es crítica.')
else:
    print('  -> Sin overfitting severo en el baseline.')

## 4. Optimización con Optuna (5-fold CV)

Se optimizan 8 hiperparámetros con 50 trials. `max_depth` acotado a [2, 6] porque con solo 7 features no tiene sentido buscar árboles profundos.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth':        trial.suggest_int('max_depth', 2, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbosity': 0,
    }
    model = XGBRegressor(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    return -scores.mean()

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nMejor RMSE CV: {study.best_value:.4f}')
print('Mejores hiperparámetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
trial_values = [t.value for t in study.trials]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(trial_values, color='steelblue', alpha=0.4, linewidth=0.8, label='Trial')
ax.plot(
    np.minimum.accumulate(trial_values),
    color='red', linewidth=1.8, label='Mejor acumulado'
)
ax.set_xlabel('Trial')
ax.set_ylabel('RMSE CV')
ax.set_title('Optuna — Convergencia de la optimización')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Modelo final con mejores hiperparámetros

In [ ]:
best_params = study.best_params.copy()
best_params.update({'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': 0})

xgb_opt = XGBRegressor(**best_params)
xgb_opt.fit(X_train, y_train)

metrics_opt = get_metrics(xgb_opt, X_train, X_test, y_train, y_test, name='XGBoost Optuna')

gap_opt = metrics_opt['RMSE_train'] - metrics_opt['RMSE_test']
print(f'\nGap RMSE tras optimización: {gap_opt:.4f}')

## 6. Importancia de features

In [ ]:
importance = pd.Series(
    xgb_opt.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('XGBoost — Importancia de features (gain)')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

print('Importancia de features (mayor a menor):')
print(importance.sort_values(ascending=False).round(4).to_string())

## 7. Diagnóstico de residuos

In [ ]:
y_pred_test = xgb_opt.predict(X_test)
residuals   = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(y_pred_test, residuals, alpha=0.4, color='steelblue', s=18)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Predicción (log €)')
axes[0].set_ylabel('Residuo')
axes[0].set_title('XGBoost Optuna — Residuos vs Predicciones')

axes[1].hist(residuals, bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de residuos')
axes[1].set_xlabel('Residuo')

stats.probplot(residuals, plot=axes[2])
axes[2].set_title('Q-Q plot')

plt.tight_layout()
plt.show()

print(f'Skewness residuos: {pd.Series(residuals).skew():.4f}')
print(f'Kurtosis residuos: {pd.Series(residuals).kurtosis():.4f}')

## 8. Error en euros (interpretabilidad)

In [ ]:
y_real_eur = np.exp(y_test.values)
y_pred_eur = np.exp(y_pred_test)
error_eur  = np.abs(y_real_eur - y_pred_eur)

print(f'Error absoluto medio  : {error_eur.mean():>10,.0f} €')
print(f'Error mediano         : {np.median(error_eur):>10,.0f} €')
print(f'Error percentil 90    : {np.percentile(error_eur, 90):>10,.0f} €')

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_real_eur, y_pred_eur, alpha=0.4, color='steelblue', s=18)
lim = [min(y_real_eur.min(), y_pred_eur.min()), max(y_real_eur.max(), y_pred_eur.max())]
ax.plot(lim, lim, 'r--', linewidth=1, label='Predicción perfecta')
ax.set_xlabel('Precio real (€)')
ax.set_ylabel('Precio predicho (€)')
ax.set_title('XGBoost Optuna — Real vs Predicho (€)')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Comparativa XGBoost baseline vs Optuna

In [ ]:
summary = pd.DataFrame({
    'XGBoost_base':   metrics_base,
    'XGBoost_Optuna': metrics_opt,
}).T.round(4)

print('Comparativa XGBoost base vs Optuna:')
display(summary)

print('\n--- Interpretación ---')
if metrics_opt['R2_test'] > metrics_base['R2_test']:
    mejora = metrics_opt['R2_test'] - metrics_base['R2_test']
    print(f'Optuna mejora R² en test en {mejora:.4f} puntos.')
else:
    print('El baseline ya era competitivo — el espacio de features limita la ganancia de optimización.')

print('\n[Para comparar con Ridge/Lasso, ver 51_linear_regression_def.ipynb]')